In [44]:
import sys
import os
import pandas as pd

# Add the project root to the Python path
sys.path.append(os.path.join(os.getcwd(), '..'))

# Reload modules when code is changed (uncomment for development)
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# 1. Project overview

What is the goal of this model?
* To see which factors have an influence on viewing figures.
* To be able to predict viewing figures.

How will the model be used?
* Program planning: when is the best time to broadcast a new program?
* Selling advertising blocks based on the number of viewers we have at certain times.

Input:
* Ratings data (top 20 Flemish programs from 2016-10-01)
* Weather, sunrise and sunset data (open-meteo data)

Which techniques do we use?
* Supervised learning: we have labels (ratings)
* Regression: we're predicting a number
* Batch learning: there is no continuous flow of new data

Performance measures:
* MAPE
* RMSE
* MAE

*We measure the distance between our prediction vector and the target values vector. In percentages for MAPE and in the number of viewers for RMSE and MAE*

# 2. Collect data

## 2.1 Ratings data

You can see how the data is collected and transformed in [src/ratings_data.py](./src/ratings_data.py).

In [45]:
ratings_df = pd.read_parquet('../data/ratings_data.parquet')

In [46]:
ratings_df.head()

,show,channel,date,start,duration,viewers
0,HET 7 UUR-JOURNAAL,EEN,2016-10-01,2016-10-01 19:00:05,0 days 00:31:38,721850
1,FC DE KAMPIOENEN,EEN,2016-10-01,2016-10-01 20:41:00,0 days 00:38:39,709606
2,WEG ZIJN WIJ,EEN,2016-10-01,2016-10-01 20:13:36,0 days 00:24:44,548239
3,IEDEREEN BEROEMD,EEN,2016-10-01,2016-10-01 19:38:10,0 days 00:29:01,523610
4,COMEDY TOPPERS,VTM,2016-10-01,2016-10-01 19:52:06,0 days 00:24:40,496216


In [47]:
ratings_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 66048 entries, 0 to 66138
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype          
---  ------    --------------  -----          
 0   show      66048 non-null  object         
 1   channel   66048 non-null  object         
 2   date      66048 non-null  datetime64[ns] 
 3   start     66048 non-null  datetime64[ns] 
 4   duration  66048 non-null  timedelta64[ns]
 5   viewers   66048 non-null  int64          
dtypes: datetime64[ns](2), int64(1), object(2), timedelta64[ns](1)
memory usage: 3.5+ MB


## 2.2 Weather data

In [48]:
weather_df = pd.read_parquet('../data/weather_data.parquet')
weather_df.head()

,date,weather_code,temperature_2m_mean,temperature_2m_max,temperature_2m_min,sunrise,sunset,daylight_duration,sunshine_duration,precipitation_sum,rain_sum,snowfall_sum,precipitation_hours,wind_speed_10m_max,wind_gusts_10m_max,sunrise_time,sunset_time
0,2016-10-01,53.0,14.094833,18.836500,10.586500,1475300633,1475342431,41795.472656,36650.582031,1.2,1.2,0.0,3.0,18.391737,45.719997,2016-10-01 07:43:53+02:00,2016-10-01 19:20:31+02:00
1,2016-10-02,53.0,12.326084,14.836500,10.336500,1475387128,1475428700,41569.531250,34481.562500,1.3,1.3,0.0,4.0,25.582806,51.480000,2016-10-02 07:45:28+02:00,2016-10-02 19:18:20+02:00
2,2016-10-03,53.0,13.478168,17.636499,10.486501,1475473624,1475514969,41342.718750,33656.269531,0.7,0.7,0.0,2.0,16.516901,36.000000,2016-10-03 07:47:04+02:00,2016-10-03 19:16:09+02:00
3,2016-10-04,3.0,12.767751,16.086500,9.836500,1475560119,1475601237,41115.175781,16417.371094,0.0,0.0,0.0,0.0,21.288757,38.160000,2016-10-04 07:48:39+02:00,2016-10-04 19:13:57+02:00
4,2016-10-05,1.0,10.753169,14.086500,8.086500,1475646615,1475687505,40887.046875,37022.519531,0.0,0.0,0.0,0.0,24.640940,48.239998,2016-10-05 07:50:15+02:00,2016-10-05 19:11:45+02:00


In [49]:
weather_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3295 entries, 0 to 3294
Data columns (total 17 columns):
 #   Column               Non-Null Count  Dtype                          
---  ------               --------------  -----                          
 0   date                 3295 non-null   datetime64[ns]                 
 1   weather_code         3295 non-null   float32                        
 2   temperature_2m_mean  3295 non-null   float32                        
 3   temperature_2m_max   3295 non-null   float32                        
 4   temperature_2m_min   3295 non-null   float32                        
 5   sunrise              3295 non-null   int64                          
 6   sunset               3295 non-null   int64                          
 7   daylight_duration    3295 non-null   float32                        
 8   sunshine_duration    3295 non-null   float32                        
 9   precipitation_sum    3295 non-null   float32                        
 10  

## 2.3 Merge ratings and weather data

In [50]:
from src.transform.data_transformer import DataTransformer

data_transformer = DataTransformer()

In [51]:
data = data_transformer.merge_ratings_and_weather_data(ratings_df, weather_df)
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 66048 entries, 0 to 66047
Data columns (total 22 columns):
 #   Column               Non-Null Count  Dtype                          
---  ------               --------------  -----                          
 0   show                 66048 non-null  object                         
 1   channel              66048 non-null  object                         
 2   date                 66048 non-null  datetime64[ns]                 
 3   start                66048 non-null  datetime64[ns]                 
 4   duration             66048 non-null  timedelta64[ns]                
 5   viewers              66048 non-null  int64                          
 6   weather_code         66048 non-null  float32                        
 7   temperature_2m_mean  66048 non-null  float32                        
 8   temperature_2m_max   66048 non-null  float32                        
 9   temperature_2m_min   66048 non-null  float32                        
 10

## 2.4 Feature engineering

In [52]:
data = data_transformer.create_new_features(data)
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 66048 entries, 0 to 66047
Data columns (total 35 columns):
 #   Column                 Non-Null Count  Dtype                          
---  ------                 --------------  -----                          
 0   show                   66048 non-null  object                         
 1   channel                66048 non-null  object                         
 2   viewers                66048 non-null  int64                          
 3   year                   66048 non-null  int32                          
 4   month                  66048 non-null  int32                          
 5   day_of_week            66048 non-null  int32                          
 6   week                   66048 non-null  UInt32                         
 7   covid_19               66048 non-null  int64                          
 8   lockdown_1             66048 non-null  int64                          
 9   lockdown_2             66048 non-null  int64      

In [53]:
data = data_transformer.select_features(data)
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 66048 entries, 0 to 66047
Data columns (total 35 columns):
 #   Column                 Non-Null Count  Dtype                          
---  ------                 --------------  -----                          
 0   show                   66048 non-null  object                         
 1   channel                66048 non-null  object                         
 2   viewers                66048 non-null  int64                          
 3   year                   66048 non-null  int32                          
 4   month                  66048 non-null  int32                          
 5   day_of_week            66048 non-null  int32                          
 6   week                   66048 non-null  UInt32                         
 7   covid_19               66048 non-null  int64                          
 8   lockdown_1             66048 non-null  int64                          
 9   lockdown_2             66048 non-null  int64      